# TTS Experiment: XTTS v2 Voice Cloning

**목적**: Coqui XTTS v2를 활용하여 특정 화자의 음성을 복제(Voice Cloning)하는 TTS 파이프라인을 구축합니다.

**시도한 내용**:
- XTTS v2 모델 로드 및 Gradio 기반 UI 실행
- 화자 음성 샘플로부터 화자 임베딩 추출 및 합성
- 한국어 텍스트 → 음성 변환 테스트

**결론**: 소량의 화자 샘플만으로 자연스러운 한국어 음성 합성이 가능함을 확인.


In [1]:
%cd /content
!git clone -b dev https://github.com/camenduru/xtts2-hf
%cd /content/xtts2-hf
!pip install -q gradio==3.50.2 TTS==0.21.1 langid unidic-lite unidic deepspeed
!pip install -q numpy<2.0.0 -U
!wget https://huggingface.co/spaces/coqui/xtts/resolve/main/examples/female.wav -O /content/xtts2-hf/examples/female.wav
!wget https://huggingface.co/spaces/coqui/xtts/resolve/main/examples/male.wav -O /content/xtts2-hf/examples/male.wav
!wget https://huggingface.co/spaces/coqui/xtts/resolve/main/ffmpeg.zip -O /content/xtts2-hf/ffmpeg.zip


/content
Cloning into 'xtts2-hf'...
remote: Enumerating objects: 518, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 518 (delta 2), reused 3 (delta 1), pack-reused 512 (from 1)
Receiving objects: 100% (518/518), 716.12 KiB | 2.34 MiB/s, done.
Resolving deltas: 100% (311/311), done.
Error downloading object: examples/female.wav (89a4fa9): Smudge error: Error downloading examples/female.wav (89a4fa9a16b6463f852cf9424f72c3d3c87aa83010e89db534c53fcd1ae12c02): [89a4fa9a16b6463f852cf9424f72c3d3c87aa83010e89db534c53fcd1ae12c02] Object does not exist on the server: [404] Object does not exist on the server

Errors logged to /content/xtts2-hf/.git/lfs/logs/20241207T085907.371287223.log
Use `git lfs logs last` to view the log.
error: external filter 'git-lfs filter-process' failed
fatal: examples/female.wav: smudge filter lfs failed
You can inspect what was checked out with 'git status'
and retry with 'git restore --source=HEAD :/'


In [2]:
!python app.py

download url: https://cotonoha-dic.s3-ap-northeast-1.amazonaws.com/unidic-3.1.0.zip
Dictionary version: 3.1.0+2021-08-31
unidic-3.1.0.zip: 100% 526M/526M [00:15<00:00, 34.8MB/s]
Finished download.
Downloaded UniDic v3.1.0+2021-08-31 to /usr/local/lib/python3.10/dist-packages/unidic/dicdir
RuntimeError: module was compiled against NumPy C-API version 0x10 (NumPy 1.23) but the running NumPy has C-API version 0xf. Check the section C-API incompatibility at the Troubleshooting ImportError section at https://numpy.org/devdocs/user/troubleshooting-importerror.html#c-api-incompatibility for indications on how to solve this problem.
2024-12-07 09:06:36.525254: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-07 09:06:36.557762: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory 

In [3]:
!pip install TTS scipy

In [4]:
!git clone https://huggingface.co/coqui/XTTS-v2

Cloning into 'XTTS-v2'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 161 (delta 69), reused 161 (delta 69), pack-reused 0 (from 0)
Receiving objects: 100% (161/161), 2.29 MiB | 10.70 MiB/s, done.
Resolving deltas: 100% (69/69), done.
Filtering content: 100% (4/4), 1.94 GiB | 32.44 MiB/s, done.


In [5]:
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

from IPython.display import Audio
from scipy.io.wavfile import write

In [6]:
config = XttsConfig()
config.load_json("./XTTS-v2/config.json")
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir="./XTTS-v2/")
model.cuda()

/usr/local/lib/python3.10/dist-packages/TTS/utils/io.py:54: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(f, map_location=map_location, **kwargs)
GPT2Infer

Xtts(
  (gpt): GPT(
    (conditioning_encoder): ConditioningEncoder(
      (init): Conv1d(80, 1024, kernel_size=(1,), stride=(1,))
      (attn): Sequential(
        (0): AttentionBlock(
          (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
          (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
          (attention): QKVAttention()
          (x_proj): Identity()
          (proj_out): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
        )
        (1): AttentionBlock(
          (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
          (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
          (attention): QKVAttention()
          (x_proj): Identity()
          (proj_out): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
        )
        (2): AttentionBlock(
          (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
          (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
          (attention): QKVAttention()
          (x_proj): Ide

In [21]:
text_to_speak = "We are going to go to the gym now to lift heavy circles."
reference_audios = ["/content/xtts2-hf/examples/male.wav"]

outputs = model.synthesize(
    text_to_speak,
    config,
    speaker_wav=reference_audios,
    gpt_cond_len=3,
    language="ko",
)

In [38]:
print("Computing speaker latents...")
gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(audio_path=reference_audios)

outputs = model.inference(
    text="도망가자 얼른",
    gpt_cond_latent=gpt_cond_latent,
    speaker_embedding=speaker_embedding,
    language="ko",
    enable_text_splitting=True
)

Computing speaker latents...


In [39]:
Audio(data=outputs['wav'], rate=24000)